In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

FIG_DIR = PLL_DIR / "figures_teammeeting"  # every figure as HTML + PNG
FIG_DIR.mkdir(exist_ok=True)

# Cherry-picked runs (results/pll)
LOOP_GAIN_CSV = PLL_DIR / "pll_test_current_max_S5_20261020_200224.csv"  # c9 min 9, 760 MHz, max x Icp, C1 7
DIV_CSV = PLL_DIR / "pll_spectrum_div_S5_20261019_174145.csv"  # 1 GHz, output divider / pad frequency varied
VCO_PNM_CSV = PLL_DIR.parent / "vco" / "vco_pnm_S5_20261021_134315.csv"  # open loop (potentiometer), PLL-matched points

INK, GRID = "#52514e", "#e4e3df"
FONT = dict(size=16)
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"), gridcolor=GRID)
N_BINS = 300  # points per plotted phase-noise curve (display only)


def traces_of(result_csv):
    """The _traces.csv of a run, only the full phase-noise curves (kind L_trace)."""
    t = pd.read_csv(result_csv.with_name(result_csv.stem + "_traces.csv"))
    return t[t["kind"] == "L_trace"]


def binned(df, f_start, f_stop):
    """Curve df (freq, level_dbm) between f_start and f_stop in N_BINS log-spaced bins (linear power mean)."""
    df = df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)].sort_values("freq")
    if len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name, width=1200):
    """HTML and PNG (kaleido) in FIG_DIR, then show the figure."""
    fig.update_layout(template="plotly_white", font=FONT)
    fig.write_html(FIG_DIR / f"{name}.html")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", width=width, height=fig.layout.height or 600, scale=2)
        print(f"saved {FIG_DIR / name}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {FIG_DIR / name}.html; PNG export failed: {e}")
    fig.show()

def settings_text(r, skip=()):
    """The setting of row `r` (a Series of a result CSV) in one line: VCO codes, f_vco, output divider and pad
    frequency, reference and loop filter. The columns in `skip` (what a figure varies) are left out."""
    def have(k):
        return k in r and k not in skip and pd.notna(r[k])

    parts = []
    codes = " ".join(f"{name} {r[k]:.0f}" for k, name in (("vco_tune_coarse", "coarse"), ("vco_current_min", "min"),
                                                            ("vco_current_max", "max")) if have(k))
    if codes:
        parts.append("VCO " + codes)
    if have("f_vco"):
        parts.append(f"f_vco {r['f_vco'] / 1e6:.0f} MHz")
    if have("total_div"):
        parts.append(f"output /{r['total_div']:.0f}, pad {r['f_vco'] / r['total_div'] / 1e6:.4g} MHz")
    if have("f_ref"):
        parts.append(f"f_ref {r['f_ref'] / 1e6:.4g} MHz")
    loop = " ".join(f"{name} {r[k]:.0f}" for k, name in (("set_current", "Icp"), ("set_c1", "C1"), ("set_c2", "C2"),
                                                          ("set_r1", "R1")) if have(k))
    if loop:
        parts.append("loop filter " + loop)
    return ", ".join(parts)


def print_settings(figures, run, r, varied, skip=()):
    """Print for the slides what `figures` show: the run, the fixed setting (row `r`, without `skip`), what varies."""
    print(f"{figures}\n  run: {run}\n  fixed: {settings_text(r, skip)}\n  varied: {varied}\n")

In [3]:
# Jitter against loop gain: c9 min 9 at 760 MHz, filter C1 7, max code x charge-pump code 1-7. set_current is
# binary weighted, so the loop gain is ~ set_current x measured |Kvco| (closed-loop VCO map). One figure for the total
# RMS jitter (full curve), one for the noise part (spurs removed); one colour per vco_current_max.
lg = pd.read_csv(LOOP_GAIN_CSV)
lg = lg[(lg["locked"] == 1) & (lg["status"] == "ok")].copy()
lg["gain"] = lg["set_current"] * lg["kvco_request"] / 1e9
lg_traces = traces_of(LOOP_GAIN_CSV)


def max_color(code):
    """Colour of a vco_current_max code (0: most current, blue; 15: least, yellow)."""
    return sample_colorscale("Viridis", [code / 15])[0]


for y, what, name in (("time_jitter_trace_ps", "Total RMS jitter", "jitter_vs_loop_gain"),
                      ("time_jitter_noise_ps", "RMS jitter of the noise part (spurs removed)",
                       "jitter_noise_vs_loop_gain")):
    fig = go.Figure()
    for code, grp in lg.groupby("vco_current_max"):
        grp = grp.sort_values("gain")
        label = f"max {code:.0f}"
        fig.add_scatter(x=grp["gain"], y=grp[y], mode="lines+markers", name=label,
                        line=dict(color=max_color(code), width=2.5), marker=dict(size=9), text=grp["set_current"],
                        hovertemplate="Icp code %{text}, gain %{x:.1f}: %{y:.0f} ps<extra>" + label + "</extra>")
    fig.update_layout(title=f"{what} vs loop gain", height=550, legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="Icp code x |Kvco| [GHz/V]", gridcolor=GRID)
    fig.update_yaxes(title="RMS jitter [ps]", gridcolor=GRID, rangemode="tozero")
    save(fig, name)
print_settings("jitter_vs_loop_gain, jitter_noise_vs_loop_gain (PLL locked)", LOOP_GAIN_CSV.name, lg.iloc[0],
               "vco_current_max 0 / 8 / 12 / 14 / 15 (one curve each) x charge-pump code 1-7 (the points); "
               "x = Icp code x |Kvco| measured at 760 MHz", skip=("vco_current_max", "set_current"))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/jitter_vs_loop_gain.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/jitter_noise_vs_loop_gain.html and .png


jitter_vs_loop_gain, jitter_noise_vs_loop_gain (PLL locked)
  run: pll_test_current_max_S5_20261020_200224.csv
  fixed: VCO coarse 9 min 9, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter C1 7 C2 0 R1 7
  varied: vco_current_max 0 / 8 / 12 / 14 / 15 (one curve each) x charge-pump code 1-7 (the points); x = Icp code x |Kvco| measured at 760 MHz



In [4]:
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]
MAX_CODES = (0, 15)  # one figure per vco_current_max

# Phase noise referred to the VCO (pad + 20 log10 N) behind the loop-gain figure: one figure per max code, colour =
# charge-pump code (light: Icp 1, dark: Icp 7).
for code in MAX_CODES:
    fig = go.Figure()
    for r in lg[lg["vco_current_max"] == code].sort_values("set_current").itertuples():
        t = lg_traces[(lg_traces["vco_current_max"] == code) & (lg_traces["set_current"] == r.set_current)]
        t = binned(t, F_start, F_stop)
        label = f"Icp {r.set_current:.0f}"
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", name=label,
                        line=dict(color=sample_colorscale("Blues", [0.3 + 0.7 * r.set_current / 7])[0], width=2),
                        hovertemplate=label + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Phase noise vs charge-pump current, max {code}", height=550,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS, range=[np.log10(F_start), np.log10(F_stop)])
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig, f"phase_noise_vs_loop_gain_max{code}")
    print_settings(f"phase_noise_vs_loop_gain_max{code} (PLL locked, L(f) referred to the VCO)", LOOP_GAIN_CSV.name,
                   lg[lg["vco_current_max"] == code].iloc[0], "charge-pump code 1-7 (one curve each)",
                   skip=("set_current",))

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/phase_noise_vs_loop_gain_max0.html and .png


phase_noise_vs_loop_gain_max0 (PLL locked, L(f) referred to the VCO)
  run: pll_test_current_max_S5_20261020_200224.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter C1 7 C2 0 R1 7
  varied: charge-pump code 1-7 (one curve each)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/phase_noise_vs_loop_gain_max15.html and .png


phase_noise_vs_loop_gain_max15 (PLL locked, L(f) referred to the VCO)
  run: pll_test_current_max_S5_20261020_200224.csv
  fixed: VCO coarse 9 min 9 max 15, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter C1 7 C2 0 R1 7
  varied: charge-pump code 1-7 (one curve each)



In [5]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]
IN_BAND = (1e3, 100e3)  # [Hz] in-band level: median of the log-binned curve over this band (robust to single peaks)

# Same VCO (1 GHz) behind every output divider, so the phase noise referred to the VCO (pad + 20 log10 N) should not
# depend on the divider. First figure: the curves, colour = pad frequency (solid: divider inside the PLL, dotted:
# + external divider). Second figure: the in-band level over IN_BAND against the pad frequency.
div = pd.read_csv(DIV_CSV)
div = div[div["status"] == "ok"].sort_values("f_out_meas").reset_index(drop=True)
div_traces = traces_of(DIV_CSV)


def pad_color(f_pad):
    """Colour of a pad frequency on a log scale between the lowest and the highest pad."""
    lo, hi = np.log10(div["f_out_meas"].min()), np.log10(div["f_out_meas"].max())
    return sample_colorscale("Plasma", [0.85 * (np.log10(f_pad) - lo) / (hi - lo)])[0]


def in_band_level(t):
    """In-band level [dBc/Hz] of the curve t: the median of its N_BINS log-spaced bins over IN_BAND, so every decade
    counts equally and a single spur or peak does not move it."""
    b = binned(t, *IN_BAND)
    return b["level_dbm"].median() if len(b) else np.nan


fig = go.Figure()
levels = []
for r in div.itertuples():
    t = div_traces[div_traces["divider"] == r.divider]
    add_db = 20 * np.log10(r.total_div)
    levels.append(in_band_level(t) + add_db)
    t = binned(t, F_start, F_stop)
    label = f"{r.divider}: pad {r.f_out_meas / 1e6:.4g} MHz"
    fig.add_scatter(x=t["freq"], y=t["level_dbm"] + add_db, mode="lines", name=label,
                    line=dict(color=pad_color(r.f_out_meas), width=2, dash="solid" if r.pll_clk_o_en == 1 else "dot"),
                    hovertemplate=label + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title="Phase noise referred to the VCO per output divider",
                  height=600, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS, range=[np.log10(F_start), np.log10(F_stop)])
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise_per_divider")

div["L_vco"] = levels
fig = go.Figure()
fig.add_scatter(x=div["f_out_meas"] / 1e6, y=div["L_vco"], mode="markers", showlegend=False,
                marker=dict(size=14, color=[pad_color(f) for f in div["f_out_meas"]],
                            symbol=["circle" if e == 1 else "circle-open" for e in div["pll_clk_o_en"]],
                            line=dict(width=2, color=[pad_color(f) for f in div["f_out_meas"]])),
                text=div["divider"], hovertemplate="%{text}: pad %{x:.4g} MHz, %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title=f"In-band phase noise ({IN_BAND[0] / 1e3:g}-{IN_BAND[1] / 1e3:g} kHz) vs pad frequency",
                  height=550)
fig.update_xaxes(title="pad frequency [MHz] (open: + external divider)", type="log", gridcolor=GRID)
fig.update_yaxes(title="in-band L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "inband_noise_vs_pad_frequency")
print_settings("phase_noise_per_divider, inband_noise_vs_pad_frequency (PLL locked, referred to the VCO)",
               DIV_CSV.name, div.iloc[0], "output divider: " + ", ".join(
                   f"/{r.total_div:.0f} {r.divider} (pad {r.f_out_meas / 1e6:.4g} MHz)" for r in div.itertuples()),
               skip=("total_div",))
div[["divider", "total_div", "f_out_meas", "L_vco", "time_jitter_trace_common_ps"]].round(2)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/phase_noise_per_divider.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/inband_noise_vs_pad_frequency.html and .png


phase_noise_per_divider, inband_noise_vs_pad_frequency (PLL locked, referred to the VCO)
  run: pll_spectrum_div_S5_20261019_174145.csv
  fixed: VCO coarse 2 min 11 max 4, f_vco 1000 MHz, f_ref 7.812 MHz, loop filter Icp 7 C1 0 C2 0 R1 7
  varied: output divider: /128 int /128 (pad 7.812 MHz), /100 ext /1 x /100 (pad 10 MHz), /64 int /64 (pad 15.62 MHz), /40 ext /4 x /10 (pad 25 MHz), /32 int /32 (pad 31.25 MHz), /16 int /16 (pad 62.5 MHz)



,divider,total_div,f_out_meas,L_vco,time_jitter_trace_common_ps
0,int /128,128,7812493.7,-79.31,55.33
1,ext /1 x /100,100,9999993.7,-70.43,102.54
2,int /64,64,15624989.5,-73.17,55.21
3,ext /4 x /10,40,24999983.2,-67.87,112.31
4,int /32,32,31249979.0,-75.79,59.53
5,int /16,16,62499955.9,-67.55,84.67


In [6]:
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]
# PLL-matched points of the open-loop run with the fewest jumps in the Phase Noise mode curve:
# (vco_tune_coarse, vco_current_min, vco_current_max, f_vco [MHz])
OPEN_CLOSED_POINTS = [(9, 9, 15, 760), (9, 9, 14, 760), (9, 8, 12, 1000), (1, 13, 15, 1000)]
CODES = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]

# Open loop (VCO_PNM_CSV: analyzer Phase Noise mode, Vctrl from a potentiometer, tuned to the PLL's f_vco) against
# the locked PLL at the same codes, divider and f_vco, both referred to the VCO (pad + 20 log10 N). One figure per
# point.
ol = pd.read_csv(VCO_PNM_CSV)
ol_traces = pd.read_csv(VCO_PNM_CSV.with_name(VCO_PNM_CSV.stem + "_traces.csv"))
ol_traces = ol_traces[ol_traces["kind"] == "L_pnmode"]
pll_curves = {}

for coarse, cur_min, cur_max, f_mhz in OPEN_CLOSED_POINTS:
    codes = dict(zip(CODES, (coarse, cur_min, cur_max)))
    r = ol[(ol["status"] == "ok") & (ol["f_vco_target"] == f_mhz * 1e6)
           & np.logical_and.reduce([ol[c] == v for c, v in codes.items()])].iloc[0]
    if r["pll_csv"] not in pll_curves:
        pll_curves[r["pll_csv"]] = traces_of(PLL_DIR / r["pll_csv"])
    t_pll = pll_curves[r["pll_csv"]]
    sel = (t_pll["f_vco"] == r["f_vco_target"]) & np.logical_and.reduce([t_pll[c] == v for c, v in codes.items()])
    for kv in r["pll_match"].split() if isinstance(r["pll_match"], str) else []:
        k, v = kv.split("=")
        sel &= t_pll[k] == float(v)
    t_ol = ol_traces[(ol_traces["f_vco_target"] == r["f_vco_target"])
                     & np.logical_and.reduce([ol_traces[c] == v for c, v in codes.items()])]

    fig = go.Figure()
    for name, t in (("open loop", t_ol), ("PLL", t_pll[sel])):
        t = binned(t, F_start, F_stop)
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines", name=name,
                        line=dict(width=2.5), hovertemplate=name + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Open vs closed loop, c{coarse} min {cur_min} max {cur_max}, {f_mhz} MHz", height=550,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS, range=[np.log10(F_start), np.log10(F_stop)])
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig, f"open_vs_closed_c{coarse}_min{cur_min}_max{cur_max}_{f_mhz}MHz")
    pll_row = pd.read_csv(PLL_DIR / r["pll_csv"])
    pll_row = pll_row[(pll_row["f_vco"] == r["f_vco_target"])
                      & np.logical_and.reduce([pll_row[c] == v for c, v in codes.items()])]
    for kv in r["pll_match"].split() if isinstance(r["pll_match"], str) else []:
        k, v = kv.split("=")
        pll_row = pll_row[pll_row[k] == float(v)]
    print_settings(f"open_vs_closed_c{coarse}_min{cur_min}_max{cur_max}_{f_mhz}MHz (L(f) referred to the VCO)",
                   f"closed loop {r['pll_csv']}, open loop {VCO_PNM_CSV.name}", pll_row.iloc[0],
                   "closed loop: the locked PLL (this setting); open loop: the free-running VCO with the same codes, "
                   "output divider and f_vco, Vctrl from a potentiometer (no loop filter, no reference)")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/open_vs_closed_c9_min9_max15_760MHz.html and .png


open_vs_closed_c9_min9_max15_760MHz (L(f) referred to the VCO)
  run: closed loop pll_test_current_max_S5_20261020_200224.csv, open loop vco_pnm_S5_20261021_134315.csv
  fixed: VCO coarse 9 min 9 max 15, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: closed loop: the locked PLL (this setting); open loop: the free-running VCO with the same codes, output divider and f_vco, Vctrl from a potentiometer (no loop filter, no reference)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/open_vs_closed_c9_min9_max14_760MHz.html and .png


open_vs_closed_c9_min9_max14_760MHz (L(f) referred to the VCO)
  run: closed loop pll_test_current_max_S5_20261020_200224.csv, open loop vco_pnm_S5_20261021_134315.csv
  fixed: VCO coarse 9 min 9 max 14, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: closed loop: the locked PLL (this setting); open loop: the free-running VCO with the same codes, output divider and f_vco, Vctrl from a potentiometer (no loop filter, no reference)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/open_vs_closed_c9_min8_max12_1000MHz.html and .png


open_vs_closed_c9_min8_max12_1000MHz (L(f) referred to the VCO)
  run: closed loop pll_test_current_max_S5_20261020_150031.csv, open loop vco_pnm_S5_20261021_134315.csv
  fixed: VCO coarse 9 min 8 max 12, f_vco 1000 MHz, output /16, pad 62.5 MHz, f_ref 7.812 MHz, loop filter Icp 7 C1 0 C2 0 R1 7
  varied: closed loop: the locked PLL (this setting); open loop: the free-running VCO with the same codes, output divider and f_vco, Vctrl from a potentiometer (no loop filter, no reference)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/open_vs_closed_c1_min13_max15_1000MHz.html and .png


open_vs_closed_c1_min13_max15_1000MHz (L(f) referred to the VCO)
  run: closed loop pll_test_current_max_S5_20261020_150031.csv, open loop vco_pnm_S5_20261021_134315.csv
  fixed: VCO coarse 1 min 13 max 15, f_vco 1000 MHz, output /16, pad 62.5 MHz, f_ref 7.812 MHz, loop filter Icp 7 C1 0 C2 0 R1 7
  varied: closed loop: the locked PLL (this setting); open loop: the free-running VCO with the same codes, output divider and f_vco, Vctrl from a potentiometer (no loop filter, no reference)



In [7]:
SHOWCASE_CSV = None  # e.g. PLL_DIR / "pll_spur_showcase_S5_20261021_180000.csv"; None = newest run
# Spectrum around the locked carrier (pll_spur_zoom.spur_showcase): both sidebands with the ~2 MHz hump and lines, in
# dBc (in the RBW) referred to the VCO: level relative to the carrier + 20 log10(N), so the spurs read their true level.
# One figure per setting and span.
if SHOWCASE_CSV is None:
    SHOWCASE_CSV = sorted((p for p in PLL_DIR.glob("pll_spur_showcase_*.csv") if not p.stem.endswith("_traces")),
                          key=lambda p: p.stat().st_mtime)[-1]
SHOWCASE_CSV = Path(SHOWCASE_CSV)
show = pd.read_csv(SHOWCASE_CSV).rename(columns={"index": "capture"})
show_traces = pd.read_csv(SHOWCASE_CSV.with_name(SHOWCASE_CSV.stem + "_traces.csv"))
show_traces = show_traces.rename(columns={"index": "capture"})
for _, row in show.iterrows():
    print_settings(f"spectrum_{row['label'].replace(' ', '_')}_span{row['span'] / 1e6:g}MHz (PLL locked, pad spectrum)",
                   SHOWCASE_CSV.name, row, f"span +-{row['span'] / 2e6:g} MHz, RBW {row['rbw']:g} Hz, "
                                           f"{row['averages']:.0f} averages; y in dBc in the RBW referred to the VCO "
                                           f"(+ 20 log10({row['total_div']:.0f}))")
for r in show.itertuples():
    t = show_traces[(show_traces["label"] == r.label) & (show_traces["capture"] == r.capture)]
    fig = go.Figure()
    dbc_vco = t["level_dbm"] - r.carrier_dbm + 20 * np.log10(r.total_div)
    fig.add_scatter(x=(t["freq"] - r.f_carrier) / 1e6, y=dbc_vco, mode="lines", line=dict(width=1, color="#2a78d6"),
                    hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"Spectrum around the carrier, {r.label} (RBW {r.rbw / 1e3:g} kHz)", height=550,
                      showlegend=False)
    fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
    fig.update_yaxes(title="level referred to the VCO [dBc]", gridcolor=GRID)
    save(fig, f"spectrum_{r.label.replace(' ', '_')}_span{r.span / 1e6:g}MHz")

spectrum_c9_min9_max0_760MHz_span6MHz (PLL locked, pad spectrum)
  run: pll_spur_showcase_S5_20261021_173106.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: span +-3 MHz, RBW 1000 Hz, 50 averages; y in dBc in the RBW referred to the VCO (+ 20 log10(8))

spectrum_c9_min9_max0_760MHz_span20MHz (PLL locked, pad spectrum)
  run: pll_spur_showcase_S5_20261021_173106.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: span +-10 MHz, RBW 3000 Hz, 30 averages; y in dBc in the RBW referred to the VCO (+ 20 log10(8))

spectrum_c9_min9_max0_760MHz_span2MHz (PLL locked, pad spectrum)
  run: pll_spur_showcase_S5_20261021_173106.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: span +-1 MHz, RBW 300 Hz, 30 averages; y in dBc in the RBW referre

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_c9_min9_max0_760MHz_span20MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_c9_min9_max0_760MHz_span2MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_c9_min14_max0_1GHz_span6MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_c9_min14_max0_1GHz_span20MHz.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_c9_min14_max0_1GHz_span2MHz.html and .png


In [8]:
VDD_CSV = PLL_DIR / "pll_test_vdd_S5_20261021_181301.csv"  # pll_test_vdd.vdd_sweep (IREF original), its _spectra
VDD_LABEL = "c9 min9 max0 760MHz"  # setting shown (the 1 GHz setting locked only at 0.75 V)
VDD_CAPTURE = 1  # capture of SHOWCASE_CAPTURES: 1 = +-3 MHz at 1 kHz, 2 = +-10 MHz at 3 kHz, 3 = +-1 MHz at 300 Hz
VDD_OVERLAY = (0.75, 0.70, 0.65, 0.60)  # [V] the supplies in the overlay figure
FLOOR_BIN, FLOOR_QUANTILE = 50e3, 0.2  # floor between the lines: lower 20 % of the points per 50 kHz

# Spectrum around the locked carrier per PLL supply (pll_test_vdd.capture_spectra: centred on the carrier, both
# sidebands, capture VDD_CAPTURE) in dBc (in the RBW) referred to the VCO, so the spurs read their true level. One
# figure per supply (same axes; 0.75 V: the first of the two), one overlay of VDD_OVERLAY, one with the floor between
# the lines of every supply (noise: in dBc/Hz referred to the VCO).
VDD_CSV = Path(VDD_CSV)
vdd_spec = pd.read_csv(VDD_CSV.with_name(VDD_CSV.stem + "_spectra.csv")).rename(columns={"vdd_set": "vdd"})
vdd_spec = vdd_spec[(vdd_spec["label"] == VDD_LABEL) & (vdd_spec["capture"] == VDD_CAPTURE)]
carriers = (vdd_spec.drop_duplicates("time").sort_values("time").drop_duplicates("vdd")
            .sort_values("vdd", ascending=False))


def vdd_spectrum(c):
    """The capture of carrier row `c`: offset from the carrier [MHz] and level referred to the VCO [dBc in the RBW]
    (level relative to the carrier + 20 log10(N))."""
    t = vdd_spec[vdd_spec["time"] == c["time"]].sort_values("freq")
    return (t["freq"] - c["f_carrier"]) / 1e6, t["level_dbm"] - c["carrier_dbm"] + 20 * np.log10(c["total_div"])


def vdd_color(v):
    return sample_colorscale("Viridis", [(v - 0.60) / 0.25 * 0.9])[0]


Y_RANGE = [-100 + 20 * np.log10(carriers["total_div"].iloc[0]),
           5 + 20 * np.log10(carriers["total_div"].iloc[0])]  # the same axes for every supply
for _, c in carriers.iterrows():
    x, y = vdd_spectrum(c)
    fig = go.Figure()
    fig.add_scatter(x=x, y=y, mode="lines", line=dict(width=1, color="#2a78d6"), showlegend=False,
                    hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
    fig.update_layout(title=f"Spectrum around the carrier, PLL supply {c['vdd']:.3f} V", height=550)
    fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
    fig.update_yaxes(title=f"level referred to the VCO [dBc in {carriers['rbw'].iloc[0]:g} Hz RBW]", gridcolor=GRID,
                     range=Y_RANGE)
    save(fig, f"spectrum_vs_vdd_{c['vdd'] * 1000:.0f}mV")

fig = go.Figure()
for _, c in carriers[carriers["vdd"].isin(VDD_OVERLAY)].iterrows():
    x, y = vdd_spectrum(c)
    fig.add_scatter(x=x, y=y, mode="lines", name=f"{c['vdd']:.2f} V", line=dict(width=1, color=vdd_color(c["vdd"])),
                    hovertemplate=f"{c['vdd']:.3f} V" + "<br>%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
fig.update_layout(title="Spectrum around the carrier vs PLL supply", height=550, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
fig.update_yaxes(title=f"level referred to the VCO [dBc in {carriers['rbw'].iloc[0]:g} Hz RBW]", gridcolor=GRID,
                 range=Y_RANGE)
save(fig, "spectrum_vs_vdd_overlay")

fig = go.Figure()
for _, c in carriers.iterrows():
    x, y = vdd_spectrum(c)
    l_vco = y.to_numpy() - 10 * np.log10(c["rbw"])  # noise: dBc/Hz
    bins = np.floor(x.to_numpy() * 1e6 / FLOOR_BIN)
    centers = [(b + 0.5) * FLOOR_BIN / 1e6 for b in np.unique(bins)]
    floors = [np.percentile(l_vco[bins == b], 100 * FLOOR_QUANTILE) for b in np.unique(bins)]
    fig.add_scatter(x=centers, y=floors, mode="lines", name=f"{c['vdd']:.3f} V",
                    line=dict(width=2, color=vdd_color(c["vdd"])),
                    hovertemplate=f"{c['vdd']:.3f} V" + "<br>%{x:.3f} MHz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title="Noise floor between the spurs vs PLL supply", height=550,
                  legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
fig.update_yaxes(title="floor referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "floor_vs_vdd")

print_settings("spectrum_vs_vdd_<mV>, spectrum_vs_vdd_overlay, floor_vs_vdd (PLL locked, pad spectrum)", VDD_CSV.name,
               carriers.iloc[0], "PLL supply " + ", ".join(f"{v:.3f}" for v in carriers["vdd"]) + " V (E36312A CH2, "
               f"set by the script); span +-{carriers['span'].iloc[0] / 2e6:g} MHz, RBW {carriers['rbw'].iloc[0]:g} "
               f"Hz, {carriers['averages'].iloc[0]:.0f} averages, centred on the carrier; spectra in dBc in the RBW "
               f"referred to the VCO (+ 20 log10({carriers['total_div'].iloc[0]:.0f})); floor = lower "
               f"{FLOOR_QUANTILE:.0%} of the points per {FLOOR_BIN / 1e3:g} kHz, in dBc/Hz referred to the VCO "
               f"(- 10 log10({carriers['rbw'].iloc[0]:g} Hz) on top)")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_850mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_800mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_750mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_700mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_675mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_650mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_625mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_600mV.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_vs_vdd_overlay.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/floor_vs_vdd.html and .png


spectrum_vs_vdd_<mV>, spectrum_vs_vdd_overlay, floor_vs_vdd (PLL locked, pad spectrum)
  run: pll_test_vdd_S5_20261021_181301.csv
  fixed: VCO coarse 9 min 9 max 0, f_vco 760 MHz, output /8, pad 95 MHz, f_ref 5.938 MHz, loop filter Icp 4 C1 7 C2 0 R1 7
  varied: PLL supply 0.850, 0.800, 0.750, 0.700, 0.675, 0.650, 0.625, 0.600 V (E36312A CH2, set by the script); span +-3 MHz, RBW 1000 Hz, 50 averages, centred on the carrier; spectra in dBc in the RBW referred to the VCO (+ 20 log10(8)); floor = lower 20% of the points per 50 kHz, in dBc/Hz referred to the VCO (- 10 log10(1000 Hz) on top)



In [ ]:
IREF_ORIGINAL_CSV = PLL_DIR / "pll_test_vdd_S5_20261021_181301.csv"  # IREF as before; its first 0.75 V rows
IREF_BATTERY_CSV = PLL_DIR / "pll_test_vdd_S5_20261022_131911.csv"  # IREF from a battery, 0.75 V
SUPPLY_BATTERY_CSV = None  # IREF and PLL supply from batteries (pll_test_vdd vdd_sweep(None)); None = newest
FILTER_TUNE_CSV = None  # pll_manual.tune_filter (battery), e.g. PLL_DIR / "pll_filter_tune_S5_...csv"; None = newest
IREF_CAPTURE = 1  # capture of SHOWCASE_CAPTURES: 1 = +-3 MHz at 1 kHz, 2 = +-10 MHz at 3 kHz, 3 = +-1 MHz at 300 Hz
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]
FILTER_KEYS = dict(icp="set_current", c1="set_c1", c2="set_c2", r1="set_r1")  # as pll_manual

# Reference current from a battery against the original IREF, both at PLL supply 0.75 V (E36312A), and IREF + PLL
# supply from batteries, at the pll_spur_zoom
# SPUR_POINTS settings: per setting the spectrum around the carrier (dBc in the RBW referred to the VCO), the Phase
# Noise mode L(f) referred to the VCO (+ the measured filters of pll_manual.tune_filter at that setting) and the RMS
# jitter (Phase Noise mode, 1 kHz-20 MHz). Settings only in the filter tuning (e.g. 1.4 GHz): L(f) and jitter per
# filter.
if FILTER_TUNE_CSV is None:
    FILTER_TUNE_CSV = max(PLL_DIR.glob("pll_filter_tune_*[0-9].csv"), key=lambda p: p.stat().st_mtime, default=None)
if SUPPLY_BATTERY_CSV is None:  # newest pll_test_vdd run after IREF_BATTERY_CSV without a recorded supply
    SUPPLY_BATTERY_CSV = max((p for p in PLL_DIR.glob("pll_test_vdd_*[0-9].csv") if p.name > IREF_BATTERY_CSV.name
                              and pd.read_csv(p)["pll_vdd"].isna().all()), default=None)


def filter_text(r):
    return " ".join(f"{k} {r[f]:.0f}" for k, f in FILTER_KEYS.items())


def pnm_runs(csv_path, vdd=None, first=False):
    """Rows of a measure_pll_vctrl CSV with a Phase Noise mode result, each with its L_pnmode curve (column "curve":
    the k-th curve of the _traces.csv belongs to the k-th such row). `vdd`: only the rows at this PLL supply [V]
    (None: all, e.g. supply from a battery); `first`: only the first row per f_vco."""
    rows = pd.read_csv(csv_path)
    rows = rows[rows["pnm_status"] == "ok"].reset_index(drop=True)
    t = pd.read_csv(csv_path.with_name(csv_path.stem + "_traces.csv"))
    t = t[t["kind"] == "L_pnmode"]
    blocks = (t["freq"].diff() < 0).cumsum().to_numpy()
    assert blocks.max() + 1 == len(rows), f"{csv_path.name}: {blocks.max() + 1} curves for {len(rows)} rows"
    rows["curve"] = [t[blocks == k][["freq", "level_dbm"]] for k in range(len(rows))]
    if vdd is not None:
        rows = rows[(rows["pll_vdd"] - vdd).abs() < 0.005]
    if first:
        rows = rows.sort_values("time").drop_duplicates("f_vco")
    return rows


def first_spectra(csv_path, vdd=0.75):
    """The capture IREF_CAPTURE of the first spectra per setting of a pll_test_vdd run at PLL supply `vdd` [V] (None:
    any, e.g. supply from a battery)."""
    s = pd.read_csv(csv_path.with_name(csv_path.stem + "_spectra.csv"))
    s = s[s["capture"] == IREF_CAPTURE]
    if vdd is not None:
        s = s[(s["vdd_set"] - vdd).abs() < 0.005]
    return s[s["time"] == s.groupby("label")["time"].transform("min")]


runs = [("IREF original", "#9a9893", pnm_runs(IREF_ORIGINAL_CSV, 0.75, True), first_spectra(IREF_ORIGINAL_CSV)),
        ("IREF battery", "#2a78d6", pnm_runs(IREF_BATTERY_CSV, 0.75, True), first_spectra(IREF_BATTERY_CSV))]
if SUPPLY_BATTERY_CSV is not None:
    runs.append(("IREF + supply battery", "#e8743b", pnm_runs(SUPPLY_BATTERY_CSV, None, True),
                 first_spectra(SUPPLY_BATTERY_CSV, None)))
codes = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]
tune = pnm_runs(FILTER_TUNE_CSV) if FILTER_TUNE_CSV is not None else pd.DataFrame(columns=["f_vco"] + codes)
table = []
for f_vco in sorted(set(runs[0][2]["f_vco"]) | set(tune["f_vco"])):
    iref = [(label, color, r[r["f_vco"] == f_vco].iloc[0], s[s["f_vco"] == f_vco].sort_values("freq"))
            for label, color, r, s in runs if (r["f_vco"] == f_vco).any()]
    ref = iref[-1][2] if iref else tune[tune["f_vco"] == f_vco].iloc[0]
    what = ("IREF original vs battery" + (" vs IREF + supply battery" if len(iref) > 2 else "") if iref
            else "loop filter")
    tag = f"{f_vco / 1e6:.0f}MHz"
    curves = [(label, color, r) for label, color, r, _ in iref]
    mine = tune[(tune["f_vco"] == f_vco) & np.logical_and.reduce([tune[c] == ref[c] for c in codes])]
    for i, (_, r) in enumerate(mine.iterrows()):
        curves.append((f"battery, {filter_text(r)}" if iref else filter_text(r),
                       sample_colorscale("Viridis", [0.9 * i / max(len(mine) - 1, 1)])[0], r))

    if iref:
        fig = go.Figure()
        for label, color, _, s in iref:
            fig.add_scatter(x=(s["freq"] - s["f_carrier"]) / 1e6,
                            y=s["level_dbm"] - s["carrier_dbm"] + 20 * np.log10(s["total_div"]), mode="lines",
                            name=label, line=dict(width=1, color=color),
                            hovertemplate=label + "<br>%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
        rbw = iref[0][3]["rbw"].iloc[0]
        fig.update_layout(title=f"Spectrum around the carrier, {what}, {f_vco / 1e6:.0f} MHz", height=550,
                          legend=dict(orientation="h", y=-0.2))
        fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
        fig.update_yaxes(title=f"level referred to the VCO [dBc in {rbw:g} Hz RBW]", gridcolor=GRID)
        save(fig, f"spectrum_iref_battery_{tag}")

    fig = go.Figure()
    for label, color, r in curves:
        t = binned(r["curve"], F_start, F_stop)
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r["total_div"]), mode="lines", name=label,
                        line=dict(width=2.5, color=color),
                        hovertemplate=label + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Phase noise, {what}, {f_vco / 1e6:.0f} MHz", height=550,
                      legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS, range=[np.log10(F_start), np.log10(F_stop)])
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig, f"phase_noise_iref_battery_{tag}" if iref else f"phase_noise_filter_{tag}")

    fig = go.Figure()
    fig.add_bar(x=[label for label, _, _ in curves], y=[r["time_jitter_pnm_ps"] for _, _, r in curves],
                marker_color=[color for _, color, _ in curves], hovertemplate="%{x}: %{y:.1f} ps<extra></extra>")
    fig.update_layout(title=f"RMS jitter, {what}, {f_vco / 1e6:.0f} MHz", height=550)
    fig.update_xaxes(gridcolor=GRID)
    fig.update_yaxes(title="RMS jitter 1 kHz-20 MHz [ps]", gridcolor=GRID, rangemode="tozero")
    save(fig, f"jitter_iref_battery_{tag}" if iref else f"jitter_filter_{tag}")

    names = (f"spectrum_iref_battery_{tag}, phase_noise_iref_battery_{tag}, jitter_iref_battery_{tag}" if iref
             else f"phase_noise_filter_{tag}, jitter_filter_{tag}")
    files = ([f"{IREF_ORIGINAL_CSV.name} (first 0.75 V)", IREF_BATTERY_CSV.name]
             + ([SUPPLY_BATTERY_CSV.name] if len(iref) > 2 else []) if iref else []) + (
        [FILTER_TUNE_CSV.name] if len(mine) else [])
    varied = ("reference current original vs battery" + (", IREF + PLL supply from batteries" if len(iref) > 2 else "")
              + (", battery IREF + other loop filters" if len(mine) else "")
              if iref else "loop filter (IREF from the battery)")
    print_settings(f"{names} (PLL locked, referred to the VCO)", ", ".join(files), iref[0][2] if iref else ref,
                   f"{varied}; PLL supply " + ("0.75 V on the E36312A CH2" if iref else f"{ref['pll_vdd']:.3f} V"
                                             if pd.notna(ref['pll_vdd']) else "from a battery")
                   + (" (IREF + supply battery: from a battery, not recorded)" if len(iref) > 2 else "")
                   + (f"; spectrum capture {IREF_CAPTURE} (RBW {rbw:g} Hz)" if iref else "")
                   + f"; L(f) and jitter from the Phase Noise mode ({ref['pnm_band']} Hz, {ref['pnm_averages']:.0f} "
                   "averages, no spur removal)", skip=tuple(FILTER_KEYS.values()) if len(mine) else ())
    for label, _, r in curves:
        table.append(dict(f_vco_MHz=f_vco / 1e6, run=label, filter=filter_text(r), jitter_ps=r["time_jitter_pnm_ps"],
                          random_ps=r["time_jitter_pnm_random_ps"], spurs_ps=r["time_jitter_pnm_discrete_ps"],
                          trace_method_ps=r["time_jitter_trace_ps"], pll_i_mA=r["pll_i"] * 1e3))
pd.DataFrame(table).round(2)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/figures_teammeeting/spectrum_iref_battery_760MHz.html and .png
